# Rubin DP2 deep-coadd IRG candidate grid (4 columns × 3 rows)

This is a **batch cutout/montage companion** to `lantern_individual_target_html_viewer_v21.ipynb`.
It retains v21's Rubin **DP2 SIA** search (`lsst.deep_coadd`), **SODA** `cutout-sync-exposure` downloads, and **IRG** rendering: **i → red, r → green, g → blue**.
The fixed asinh display stretch and saturation are the same as v21 by default.

**Two input modes (choose exactly one):**

- `TARGET_IDS`: list of Second-Pass `target_id`s, resolved to `ra`, `dec` in `second_pass_candidates_with_flags.parquet`.
- `COORDINATES`: list of `(ra_deg, dec_deg)` pairs; optional `(label, ra_deg, dec_deg)` entries. **No local catalog needed**.

**Output:** one Matplotlib PNG per **12 targets (4 columns, 3 rows)**, one self-contained HTML montage with all target panels, a CSV status table, and cached coadd arrays for efficient reruns.

**Execution environment:** run this in a **Rubin Science Platform (RSP)** Python environment with authenticated access to **DP2** and the same `lsst.rsp` / `lsst.images` libraries used by v21. Nothing is fetched until the retrieval cell runs. Cutouts can fail outside the DP2 footprint or if services/authentication are unavailable.

This notebook **does not** fetch ANTARES alert thumbnails, PSFs, or deconvolution images. A band substitution is shown explicitly; disable it if you require genuine `irg` coverage.

In [ ]:
# ============================================================
# USER SETTINGS — edit this cell
# ============================================================

from pathlib import Path

# OPTION A: use target IDs (from the same candidates file as v21).
TARGET_IDS = [
    # "target_1",
    # "target_2",
    "target_0002330",
    "target_0005448",
    "target_0012511",
    "target_0024385",
    "target_0010393",
    "target_0000641",
    "target_0012310",
    "target_0030455",
    "target_0020630",   
]

TARGET_IDS0 = [
    # "target_1",
    # "target_2",
    "target_0002330",
    "target_0002938",
    "target_0005448",
    "target_0009843",
    "target_0024750",
    "target_0029791",
    "target_0033124",
    "target_0034363",
    "target_0008756",
    "target_0025823",
    "target_0012511",
    "target_0024385",
    "target_0009563",
    "target_0010393",
]

TARGET_IDS1 = [
    "target_0039816",
    "target_0011102",
    "target_0000641",
    "target_0010498",
    "target_0012310",
    "target_0010654",
    "target_0024105",
    "target_0030455",
    "target_0015373",
    "target_0020630",
    "target_0005487",
    "target_0007636",
    "target_0011540",
    "target_0029368",
    "target_0039871",
    "target_0005743",
    "target_0030276",
    "target_0008107",
    "target_0031536",
    "target_0008859",
    "target_0009557",
]

# OPTION B: use positions (decimal degrees, ICRS).
# You may provide (ra, dec) or (label, ra, dec).
COORDINATES = [
    # (53.123456, -27.123456),
    # ("candidate B", 53.234567, -27.234567),
]

CANDIDATES_FILE = Path(
    "candidate_analysis_hybrid_multi_loci_v8_no_gaia_color_cut/"
    "second_pass_candidates_with_flags.parquet"
)

OUTPUT_DIR = Path("rubin_dp2_irg_grid")
CACHE_DIR = OUTPUT_DIR / "coadd_cache"
USE_CACHE = True

N_COLUMNS = 3 #4
N_ROWS = 3
PER_PAGE = N_COLUMNS * N_ROWS   # 12 candidates per figure

# Same DP2 cutout radius as v21: 0.001 deg = 3.6 arcsec radius.
COADD_CUTOUT_RADIUS_DEG = 0.001
STAMP_SIZE_PIX = 32           # fixed square stamp size, pixel scale ~0.2"/pix

# v21 fixed display stretch. Use the SAME normalization for every target.
ASINH_A = 0.002
VMIN = -0.03
VMAX = 3000.0
RGB_SATURATION = 1.6

# v21 substitutes the nearest available band if needed.
# Set False to require true g, r, i bands for each RGB cutout.
ALLOW_BAND_FALLBACK = True

SAVE_HTML = True
SAVE_PNG = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ============================================================
# IMPORTS — use a Rubin RSP Python environment
# ============================================================

import base64
import hashlib
import html
import io
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance

from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.visualization import AsinhStretch, ImageNormalize

from lsst.images.serialization import read_archive
from lsst.rsp import RSPDiscovery
from lsst.rsp.utils import get_pyvo_auth
from pyvo.dal.adhoc import SodaQuery

from IPython.display import display, HTML

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 110,
})

## 1. Resolve target IDs or RA/Dec pairs

The `target_id` lookup uses v21's candidate Parquet input. IDs are matched **as strings** and stay in the requested order. Invalid IDs or coordinates are retained in the grid as error panels instead of silently disappearing.

In [ ]:
def prepare_targets():
    if bool(TARGET_IDS) == bool(COORDINATES):
        raise ValueError("Fill exactly one of TARGET_IDS or COORDINATES.")

    rows = []

    if TARGET_IDS:
        if not CANDIDATES_FILE.exists():
            raise FileNotFoundError(f"Missing candidates file: {CANDIDATES_FILE}")

        candidates = pd.read_parquet(
            CANDIDATES_FILE,
            columns=["target_id", "ra", "dec"],
        ).copy()
        candidates["_id"] = candidates["target_id"].astype(str)
        groups = {key: group for key, group in candidates.groupby("_id", sort=False)}

        for target_id in TARGET_IDS:
            label = str(target_id)
            matches = groups.get(label)
            if matches is None or len(matches) != 1:
                reason = ("ID not found" if matches is None
                          else f"ID is ambiguous: {len(matches)} rows")
                rows.append({"label": label, "target_id": label,
                             "ra": np.nan, "dec": np.nan, "input_error": reason})
            else:
                m = matches.iloc[0]
                rows.append({"label": label, "target_id": label,
                             "ra": m["ra"], "dec": m["dec"], "input_error": ""})

    else:
        for number, entry in enumerate(COORDINATES, start=1):
            if len(entry) == 2:
                ra, dec = entry
                label = f"coord_{number:03d}"
            elif len(entry) == 3:
                label, ra, dec = entry
            else:
                raise ValueError(f"COORDINATES entry #{number} needs 2 or 3 fields: {entry}")
            rows.append({"label": str(label), "target_id": "",
                         "ra": ra, "dec": dec, "input_error": ""})

    for row in rows:
        try:
            ra = float(row["ra"])
            dec = float(row["dec"])
            if not (np.isfinite(ra) and np.isfinite(dec)
                    and 0 <= ra < 360 and -90 <= dec <= 90):
                raise ValueError("RA/Dec out of range or nonfinite")
            row["ra"], row["dec"] = ra, dec
        except (ValueError, TypeError) as exc:
            row["ra"], row["dec"] = np.nan, np.nan
            row["input_error"] = row["input_error"] or str(exc)

    return rows


targets = prepare_targets()
print(f"Requested {len(targets)} objects = {(len(targets) + PER_PAGE - 1) // PER_PAGE} page(s).")
display(pd.DataFrame(targets))

In [ ]:
for target in targets:
    print("%s, %.4f, %.4f"%(target['target_id'], target['ra'], target['dec']) )

## 2. Connect to Rubin DP2

Uses the same two RSP calls as v21. You must be in an RSP environment with access to DP2. The first cutout request is made in the following section, not here.

In [ ]:
discovery = RSPDiscovery("dp2")
sia_client = discovery.get_sia_client()
print("DP2 SIA client initialized.")

## 3. Fetch and cache IRG deep-coadd cutouts

For each position, one SIA query finds DP2 deep-coadd exposure products. Only the bands needed for IRG are downloaded by default. If a needed band fails and `ALLOW_BAND_FALLBACK=True`, the next-nearest successful band in `ugrizy` is tried, as in v21. **Actual band mappings are saved and labeled.**

The images are cropped around the sky position to the same **32 × 32** shape using each exposure's sky-to-pixel transformation. This centers each cutout without resizing the native coadd pixels. The three coadd bands are presumed to share their local pixel orientation (as normally expected for coadds); this is **not** a general WCS reprojection across different coadd patches.

In [ ]:
BAND_ORDER = "ugrizy"
RGB_BANDS = ("i", "r", "g")  # R, G, B channels
BAND_INDEX = {band: i for i, band in enumerate(BAND_ORDER)}
BAND_CENTER_INDEX = (len(BAND_ORDER) - 1) / 2


def ranked_source_bands(requested, available):
    available = [band for band in BAND_ORDER if band in set(available)]
    if not ALLOW_BAND_FALLBACK:
        return [requested] if requested in available else []
    return sorted(
        available,
        key=lambda band: (
            abs(BAND_INDEX[band] - BAND_INDEX[requested]),
            abs(BAND_INDEX[band] - BAND_CENTER_INDEX),
            BAND_INDEX[band],
        ),
    )


def make_centered_stamp(exposure, ra, dec, size=STAMP_SIZE_PIX):
    image = np.asarray(exposure.image.array, dtype=float)
    if image.ndim != 2:
        raise ValueError(f"Exposure array must be 2D, got {image.shape}.")

    # v21's sky_projection transformation, now used to center every band.
    coord = SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs")
    try:
        xy = exposure.sky_projection.sky_to_pixel(coord)
        x_center = float(xy.x) - float(exposure.bbox.x.min)
        y_center = float(xy.y) - float(exposure.bbox.y.min)
        if not (np.isfinite(x_center) and np.isfinite(y_center)):
            raise ValueError("Nonfinite sky-to-pixel result")
    except Exception:
        # As in v21, fall back to the cutout center if WCS is unavailable.
        y_center = (image.shape[0] - 1) / 2
        x_center = (image.shape[1] - 1) / 2

    x0 = int(round(x_center - (size - 1) / 2))
    y0 = int(round(y_center - (size - 1) / 2))
    x1, y1 = x0 + size, y0 + size
    height, width = image.shape

    sx0, sx1 = max(0, x0), min(width, x1)
    sy0, sy1 = max(0, y0), min(height, y1)
    stamp = np.full((size, size), np.nan, dtype=float)
    if sx0 < sx1 and sy0 < sy1:
        stamp[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = image[sy0:sy1, sx0:sx1]
    if not np.any(np.isfinite(stamp)):
        raise RuntimeError("Centered stamp contains no finite pixels")
    return stamp


def candidate_indices_for_band(table, band, ra, dec):
    names = table.colnames
    indices = np.flatnonzero(np.asarray(table["lsst_band"]).astype(str) == band)
    if len(indices) > 1 and "s_ra" in names and "s_dec" in names:
        try:
            coords = SkyCoord(
                ra=np.asarray(table["s_ra"])[indices].astype(float) * u.deg,
                dec=np.asarray(table["s_dec"])[indices].astype(float) * u.deg,
            )
            target = SkyCoord(ra=ra * u.deg, dec=dec * u.deg)
            indices = indices[np.argsort(coords.separation(target).deg)]
        except Exception:
            pass
    return indices


def fetch_dp2_irg(ra, dec):
    """Return a 3×N×N array in IRG order and the actual source-band mapping."""
    search = sia_client.search(
        pos=(ra, dec, COADD_CUTOUT_RADIUS_DEG),
        calib_level=3,
        dpsubtype="lsst.deep_coadd",
    )
    table = search.to_table()
    if len(table) == 0 or "lsst_band" not in table.colnames:
        raise RuntimeError("No DP2 deep-coadd results at this position")

    available = set(np.asarray(table["lsst_band"]).astype(str)) & set(BAND_ORDER)
    if not available:
        raise RuntimeError("No usable ugrizy bands reported by DP2 SIA")

    fetched = {}       # band -> np.ndarray, or None if failed
    fetch_errors = {}  # band -> error message

    def get_one_band(band):
        if band in fetched:
            return fetched[band]
        errors = []
        for index in candidate_indices_for_band(table, band, ra, dec):
            try:
                link = discovery.get_datalink_results(search[int(index)])
                soda = SodaQuery.from_resource(
                    link,
                    link.get_adhocservice_by_id("cutout-sync-exposure"),
                    session=get_pyvo_auth(),
                )
                soda.circle = (ra * u.deg, dec * u.deg,
                               COADD_CUTOUT_RADIUS_DEG * u.deg)
                payload = soda.execute_stream().read()
                soda.raise_if_error()
                exposure = read_archive(io.BytesIO(payload))
                fetched[band] = make_centered_stamp(exposure, ra, dec)
                return fetched[band]
            except Exception as exc:
                errors.append(f"SIA row {int(index)}: {exc}")
        fetched[band] = None
        fetch_errors[band] = " | ".join(errors) if errors else "no SIA rows"
        return None

    mapping = {}
    layers = []
    for requested in RGB_BANDS:
        for possible_band in ranked_source_bands(requested, available):
            stamp = get_one_band(possible_band)
            if stamp is not None:
                mapping[requested] = possible_band
                layers.append(stamp)
                break
        else:
            raise RuntimeError(
                f"Cannot obtain image for {requested} channel. "
                f"Band errors: {fetch_errors}"
            )

    return np.stack(layers, axis=0), mapping


def cache_file_for(ra, dec):
    key = (f"{ra:.9f}/{dec:.9f}/{COADD_CUTOUT_RADIUS_DEG:.9f}/"
           f"{STAMP_SIZE_PIX}/{int(ALLOW_BAND_FALLBACK)}")
    name = hashlib.sha256(key.encode("utf-8")).hexdigest()[:20]
    return CACHE_DIR / f"deep_coadd_irg_{name}.npz"


def load_or_fetch(ra, dec):
    path = cache_file_for(ra, dec)
    if USE_CACHE and path.exists():
        try:
            with np.load(path, allow_pickle=False) as data:
                layers = np.asarray(data["layers"], dtype=float)
                mapping = {band: str(data[f"source_{band}"].item())
                           for band in RGB_BANDS}
            if layers.shape == (3, STAMP_SIZE_PIX, STAMP_SIZE_PIX):
                return layers, mapping, True
        except Exception as exc:
            print(f"Cache unreadable ({path.name}); refetching: {exc}")

    layers, mapping = fetch_dp2_irg(ra, dec)
    if USE_CACHE:
        np.savez_compressed(
            path,
            layers=layers,
            **{f"source_{band}": np.asarray(mapping[band]) for band in RGB_BANDS},
        )
    return layers, mapping, False

## 4. Convert i/r/g arrays to v21-style colors and retrieve the targets

All candidates use v21's shared, fixed `vmin`, `vmax`, and `AsinhStretch(a=0.002)` by default. Missing pixels are black. Retrieval errors are recorded per target and will appear as empty/error panels rather than terminating the whole batch.

In [ ]:
def normalize_science_array(data):
    # v21's exact fixed stretch, with NaNs made safe for RGB conversion.
    data = np.asarray(data, dtype=float)
    safe = np.nan_to_num(data, nan=VMIN, posinf=VMAX, neginf=VMIN)
    normalize = ImageNormalize(
        safe,
        vmin=VMIN,
        vmax=VMAX,
        stretch=AsinhStretch(a=ASINH_A),
        clip=True,
    )
    return np.asarray(normalize(safe), dtype=float)


def make_irg_rgb(layers):
    if layers.shape != (3, STAMP_SIZE_PIX, STAMP_SIZE_PIX):
        raise ValueError(f"Expected 3×{STAMP_SIZE_PIX}×{STAMP_SIZE_PIX}, got {layers.shape}")
    rgb = np.dstack([normalize_science_array(layer) for layer in layers])
    rgb8 = np.uint8(np.rint(255 * np.clip(rgb, 0, 1)))
    pil = Image.fromarray(rgb8, mode="RGB")
    if RGB_SATURATION != 1.0:
        pil = ImageEnhance.Color(pil).enhance(RGB_SATURATION)
    # Native array orientation. In matplotlib use origin='lower'.
    return np.asarray(pil).astype(float) / 255.0


results = []
for index, t in enumerate(targets, start=1):
    item = dict(t)
    item.update(status="error", error="", mapping={}, rgb=None, cached=False)
    print(f"[{index}/{len(targets)}] {t['label']}", end=" ")
    try:
        if t["input_error"]:
            raise ValueError(t["input_error"])
        layers, mapping, cached = load_or_fetch(t["ra"], t["dec"])
        item.update(status="ok", rgb=make_irg_rgb(layers),
                    mapping=mapping, cached=cached)
        note = ", ".join(f"{b}->{mapping[b]}" for b in RGB_BANDS)
        print(f"OK ({note})" + (" [cache]" if cached else ""))
    except Exception as exc:
        item["error"] = str(exc)
        print("FAILED:", exc)
    results.append(item)

summary = pd.DataFrame([{
    "label": item["label"],
    "target_id": item["target_id"],
    "ra": item["ra"],
    "dec": item["dec"],
    "status": item["status"],
    "i_source": item["mapping"].get("i", ""),
    "r_source": item["mapping"].get("r", ""),
    "g_source": item["mapping"].get("g", ""),
    "used_fallback": any(item["mapping"].get(b) not in (None, b) for b in RGB_BANDS),
    "from_cache": item["cached"],
    "error": item["error"],
} for item in results])

csv_path = OUTPUT_DIR / "rubin_dp2_irg_grid_status.csv"
summary.to_csv(csv_path, index=False)
print(f"\nSuccessful: {(summary.status == 'ok').sum()} / {len(summary)}")
print("Status CSV:", csv_path)
display(summary)

## 5. Render the 4 × 3 Matplotlib grids

By default, 4 **columns** × 3 **rows**. Each tile displays one candidate with its ID (or coordinate label), position, and any substituted band. Additional sets of 12 automatically produce new pages. Figures are displayed inline and saved as high-resolution PNG.

In [ ]:
def compact_error(text, max_chars=100):
    return text if len(text) <= max_chars else text[:max_chars - 3] + "..."


saved_pngs = []
for page_start in range(0, len(results), PER_PAGE):
    subset = results[page_start:page_start + PER_PAGE]
    page_no = page_start // PER_PAGE + 1
    fig, axes = plt.subplots(
        N_ROWS, N_COLUMNS,
        figsize=(9,9), #(12.0, 9.6),
        squeeze=False,
        constrained_layout=True,
    )
    for ax in axes.flat:
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_visible(False)

    for offset, item in enumerate(subset):
        ax = axes.flat[offset]
        ax.set_visible(True)
        global_number = page_start + offset + 1
        label = item["label"]
        if len(label) > 31:
            label = label[:28] + "..."
        #ax.set_title(f"{global_number}. {label}", fontsize=11, pad=5)
        if item["status"] == "ok":
            ax.imshow(item["rgb"], origin="lower", interpolation="nearest",
                      vmin=0, vmax=1)
            if any(item["mapping"][b] != b for b in RGB_BANDS):
                fallback = " ".join(
                    f"{b}→{item['mapping'][b]}" for b in RGB_BANDS
                    if item["mapping"][b] != b
                )
                ax.text(0.5, 0.05, fallback, transform=ax.transAxes,
                        ha="center", va="bottom", fontsize=10, color="white",
                        bbox={"facecolor": "black", "alpha": 0.7, "pad": 2})
        else:
            ax.set_facecolor("0.9")
            ax.text(0.5, 0.5, "Cutout unavailable\n" + compact_error(item["error"]),
                    transform=ax.transAxes, ha="center", va="center",
                    fontsize=9, wrap=True)

        #if np.isfinite(item["ra"]) and np.isfinite(item["dec"]):
            #ax.set_xlabel(f"{item['ra']:.5f}, {item['dec']:+.5f} deg", fontsize=10)

    #fig.suptitle(f"Rubin DP2 deep coadd · IRG  |  page {page_no}", fontsize=14)
    if SAVE_PNG:
        filename = OUTPUT_DIR / f"rubin_dp2_irg_grid_page_{page_no:02d}.png"
        fig.savefig(filename, dpi=170, bbox_inches="tight")
        saved_pngs.append(filename)
        print("Saved:", filename)
    plt.show()
    plt.close(fig)

## 6. Save a self-contained HTML grid (optional)

Unlike the individual-target v21 HTML, this HTML page is a lightweight *gallery* with only the DP2 IRG panels. It embeds the image pixels directly, so it can be opened without logging into Rubin again. Images use **nearest-neighbor pixel display**, as in v21.

In [ ]:
def rgb_to_png_data_uri(rgb):
    image = Image.fromarray(
        np.uint8(np.rint(255 * np.clip(np.flipud(rgb), 0, 1))), mode="RGB"
    )
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode("ascii")


if SAVE_HTML:
    all_pages = []
    for page_start in range(0, len(results), PER_PAGE):
        page_number = page_start // PER_PAGE + 1
        cards = []
        for item in results[page_start:page_start + PER_PAGE]:
            label = html.escape(item["label"])
            if np.isfinite(item["ra"]) and np.isfinite(item["dec"]):
                coords = f"RA {item['ra']:.6f}°, Dec {item['dec']:+.6f}°"
            else:
                coords = "Position unavailable"
            if item["status"] == "ok":
                src = rgb_to_png_data_uri(item["rgb"])
                image_markup = f'<img alt="IRG cutout for {label}" src="{src}">'
                substitutions = ", ".join(
                    f"{b}→{item['mapping'][b]}" for b in RGB_BANDS
                    if item["mapping"][b] != b
                )
                warning = (f'<div class="fallback">Fallback: {html.escape(substitutions)}</div>'
                           if substitutions else "")
            else:
                image_markup = '<div class="missing">Cutout unavailable</div>'
                warning = f'<div class="error">{html.escape(compact_error(item["error"], 300))}</div>'
            cards.append(
                f'<article><h3>{label}</h3>{image_markup}'
                f'<div class="coords">{html.escape(coords)}</div>{warning}</article>'
            )
        all_pages.append(
            f'<section class="page"><h2>Page {page_number}</h2>'
            f'<div class="grid">{"".join(cards)}</div></section>'
        )

    markup = """<!doctype html>
<html lang="en">
<head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Rubin DP2 IRG candidate gallery</title>
<style>
body{font-family:Arial,Helvetica,sans-serif;max-width:1400px;margin:20px auto;padding:0 16px;background:#f8f9fa;color:#20252c}
h1{font-size:1.6rem;margin-bottom:6px}.hint{color:#565d67;font-size:.95rem}
.page{margin:24px 0 40px;break-after:page}h2{font-size:1.2rem}
.grid{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px}
article{background:white;border:1px solid #ccd0d5;border-radius:6px;padding:10px;min-width:0}
h3{font-size:1rem;margin:2px 0 8px;overflow-wrap:anywhere}
article img{display:block;width:100%;height:auto;aspect-ratio:1;object-fit:contain;image-rendering:pixelated;background:black}
.coords,.fallback,.error{font-size:.83rem;margin-top:7px;overflow-wrap:anywhere}
.coords{color:#515966}.fallback{color:#815500}.error{color:#9e2828}
.missing{height:auto;aspect-ratio:1;display:grid;place-items:center;background:#e8eaed;color:#555}
@media(max-width:750px){.grid{grid-template-columns:repeat(2,minmax(0,1fr))}}
@media print{body{background:white}article{break-inside:avoid}}
</style></head>
<body><h1>Rubin DP2 deep-coadd candidate IRG gallery</h1>
<p class="hint">i = red, r = green, g = blue. Fixed v21 asinh stretch; original coadd pixels are preserved.</p>
{pages}
</body></html>""".replace("{pages}", "\n".join(all_pages))

    html_path = OUTPUT_DIR / "rubin_dp2_irg_grid.html"
    html_path.write_text(markup, encoding="utf-8")
    print("Saved HTML:", html_path)
else:
    print("SAVE_HTML=False")

## Usage and caveats

1. Fill **either** `TARGET_IDS` **or** `COORDINATES` in the first code cell; run cells in order on Rubin RSP.
2. To change only the display stretch or saturation, edit the display settings and rerun sections 4–6; cached pixel cutouts avoid repeat DP2 requests when rerunning from section 3.
3. Each 32-pixel stamp spans roughly 6.4 arcsec assuming the standard ~0.2 arcsec/pixel Rubin scale; it is centered on the input coordinate (subject to pixel rounding).
4. The RGB uses *the original v21 fixed asinh normalization*, not a per-target dynamic stretch. That supports consistent visual comparison, but faint candidates may benefit from changes to `VMAX`, `VMIN`, or `ASINH_A`.
5. **Band substitutions are explicitly annotated**. To prevent misleading composites when a true `i`, `r`, or `g` band is missing, use `ALLOW_BAND_FALLBACK=False`.
6. The three bands are centered on the input sky location but not reprojected onto a new common WCS. Usually DP2 deep-coadd bands align; manually verify registration near tract/patch boundaries.
7. This is a **visual inspection montage**, not calibrated photometry or a lens-confirmation test.
8. Delete the relevant `.npz` cache file(s), or set `USE_CACHE=False`, to force new DP2 downloads after changing server-side products.